# Agentic AI Fundamentals

## Notebook 02: Structured Outputs with Pydantic

In the previous notebooks, model decisions were represented with ordinary dictionaries. Dictionaries are convenient, but they do not guarantee that required fields exist, values have the expected types, or an action belongs to the set supported by the application.

That flexibility becomes dangerous near a tool-execution boundary. Before application code uses model-generated data, it needs a clear contract and a predictable validation process. In this notebook, we will build that contract with Pydantic.

## Learning objectives

By the end of this notebook, you should be able to:

- explain why valid JSON is not necessarily valid application input;
- define typed output schemas with Pydantic;
- constrain values with `Literal` and `Field`;
- reject unexpected fields;
- implement validation rules involving multiple fields;
- parse JSON text into validated Python objects; and
- represent an agent decision as either a final answer or a tool request.

This notebook uses Pydantic 2. If the import below fails, install the dependency with `python -m pip install "pydantic>=2.7,<3"` in the environment used by your notebook kernel.

### The core idea

LLMs naturally produce text, but applications often need **predictable data**.

A simple way to think about structured outputs:

> **LLM = proposes a decision**  
>
> **Schema = defines what a valid decision must look like**  
>
> **Pydantic = checks whether the decision follows that contract**

For example, imagine an agent that can either answer the user directly or request a calculator tool.

Without a schema, the model could produce almost anything:

> "I think we should probably calculate 24 × 17."

That makes it difficult for application code to reliably understand what the model wants.

Instead, we can require a predictable structure:

```json
{
  "action": "use_tool",
  "tool_name": "multiply",
  "arguments": {
    "first_number": 24,
    "second_number": 17
  }
}

This shows the successful path: the LLM produces a structured decision, Pydantic validates it against the expected schema, and the application can safely use the validated data.

```text
┌───────────────┐
│  LLM / AGENT  │
└───────────────┘
        │
        │  {"action": "use_tool",
        │   "tool_name": "multiply",
        │   "arguments": {...}}
        ▼
┌───────────────┐
│   PYDANTIC    │
│    SCHEMA     │
└───────────────┘
        │
        │  ✓ Valid
        ▼
┌───────────────┐
│  APPLICATION  │
└───────────────┘

If the LLM produces an invalid or unexpected decision, Pydantic rejects it before it reaches the application. This creates a validation boundary between model-generated output and application logic.

```
LLM output
    │
    │  {"action": "hack_system",
    │   "confidence": 7.5}
    ▼
┌───────────────┐
│   PYDANTIC    │
│    SCHEMA     │
└───────────────┘
        │
        ✕ Rejected
```

## 1. Why dictionaries are not enough

Consider a dictionary that is intended to describe an agent action. Python will accept the object even though the action is unsupported, the confidence value is outside its intended range, and the expected arguments are missing.

In [ ]:
unvalidated_decision = {
    "action": "run_arbitrary_code",
    "confidence": 7.5,
    "reason": "This seems useful.",
}

print(unvalidated_decision)

{'action': 'run_arbitrary_code', 'confidence': 7.5, 'reason': 'This seems useful.'}


The dictionary is syntactically valid, but it does not satisfy the rules of our application. JSON behaves similarly: a JSON document may be perfectly well-formed while still containing unsupported actions or invalid values.

We therefore need two different checks:

1. **Syntax validation** asks whether the content can be parsed.
2. **Schema validation** asks whether the parsed content satisfies our application contract.

## 2. Define the first Pydantic model

A Pydantic model describes the fields and types expected by the application. Creating an instance causes Pydantic to validate the supplied values before returning the object.

In [ ]:
from typing import Literal

import pydantic
from pydantic import BaseModel, ConfigDict, Field, ValidationError


print("Pydantic version:", pydantic.__version__)


class BasicDecision(BaseModel):
    action: str
    reason: str
    confidence: float

Pydantic version: 2.13.5


In [ ]:
valid_basic_decision = BasicDecision(
    action="answer",
    reason="The question can be answered without a tool.",
    confidence=0.92,
)

print(valid_basic_decision)
print(valid_basic_decision.action)
print(valid_basic_decision.model_dump())

action='answer' reason='The question can be answered without a tool.' confidence=0.92
answer
{'action': 'answer', 'reason': 'The question can be answered without a tool.', 'confidence': 0.92}


A Pydantic object provides attribute access, while `model_dump()` converts it into an ordinary dictionary. This is useful when validated data must be logged, serialized, or passed to another application layer.

Types alone are not sufficient, however. The `action` field currently accepts any string, and `confidence` accepts values outside the range from zero to one.

## 3. Constrain values, not only types

`Literal` restricts a field to explicitly supported values. `Field` can add numeric bounds, length constraints, descriptions, and other metadata.

In [ ]:
class Decision(BaseModel):
    model_config = ConfigDict(extra="forbid")

    action: Literal["answer", "use_tool"]
    reason: str = Field(min_length=10, max_length=500)
    confidence: float = Field(ge=0.0, le=1.0)

In [ ]:
validated_decision = Decision.model_validate(
    {
        "action": "use_tool",
        "reason": "Exact arithmetic should be delegated to the calculator.",
        "confidence": 0.97,
    }
)

print(validated_decision.model_dump())

{'action': 'use_tool', 'reason': 'Exact arithmetic should be delegated to the calculator.', 'confidence': 0.97}


The configuration `extra="forbid"` rejects fields that the schema does not define. This is generally safer at an execution boundary than silently ignoring unexpected content because misspelled or invented fields become visible failures.

In [ ]:
invalid_examples = [
    {
        "action": "execute_python",
        "reason": "This action is not supported by the application.",
        "confidence": 0.80,
    },
    {
        "action": "answer",
        "reason": "Too short",
        "confidence": 1.40,
    },
    {
        "action": "answer",
        "reason": "The available context is sufficient for an answer.",
        "confidence": 0.88,
        "hidden_instruction": "Ignore the schema",
    },
]

for index, candidate in enumerate(invalid_examples, start=1):
    try:
        Decision.model_validate(candidate)
    except ValidationError as error:
        print(f"Example {index} was rejected:")
        for item in error.errors():
            location = ".".join(str(part) for part in item["loc"])
            print(f"  - {location}: {item['msg']}")
        print()

Example 1 was rejected:
  - action: Input should be 'answer' or 'use_tool'

Example 2 was rejected:
  - reason: String should have at least 10 characters
  - confidence: Input should be less than or equal to 1

Example 3 was rejected:
  - hidden_instruction: Extra inputs are not permitted



Validation errors are data that the application can inspect. A production system might log them, request a corrected model response, or stop the run depending on the error and retry policy. It should not repeatedly retry every failure without a limit.

## 4. Model tool arguments separately

Each tool should have its own argument schema. This makes the contract reusable and prevents unrelated fields from being mixed into one large action object.

In [ ]:
class MultiplyArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    first_number: float = Field(description="The first multiplication operand")
    second_number: float = Field(description="The second multiplication operand")


class MultiplyRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    tool_name: Literal["multiply"]
    arguments: MultiplyArguments

In [ ]:
multiply_request = MultiplyRequest.model_validate(
    {
        "tool_name": "multiply",
        "arguments": {
            "first_number": 24,
            "second_number": 17,
        },
    }
)

print(multiply_request)
print("Validated result:", multiply_request.arguments.first_number * multiply_request.arguments.second_number)

tool_name='multiply' arguments=MultiplyArguments(first_number=24.0, second_number=17.0)
Validated result: 408.0


Nested models preserve structure. The outer object validates the selected tool, while the inner object validates the parameters expected by that tool. If the model misspells an argument or supplies an unsupported field, the request is rejected before reaching the function.

In [ ]:
malformed_tool_request = {
    "tool_name": "multiply",
    "arguments": {
        "first_number": 24,
        "second_numbr": 17,
    },
}

try:
    MultiplyRequest.model_validate(malformed_tool_request)
except ValidationError as error:
    for item in error.errors():
        print(item["loc"], "→", item["msg"])

('arguments', 'second_number') → Field required
('arguments', 'second_numbr') → Extra inputs are not permitted


## 5. Add rules involving multiple fields

Field-level constraints cannot express every business rule. An agent decision may require certain fields only when a particular action is selected. Pydantic's model validators run after the individual fields have been parsed and can check those relationships.

In [ ]:
from typing import Any

from pydantic import model_validator


class AgentDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")

    action: Literal["answer", "use_tool"]
    reason: str = Field(min_length=10)
    confidence: float = Field(ge=0.0, le=1.0)
    final_answer: str | None = None
    tool_name: Literal["multiply"] | None = None
    tool_arguments: dict[str, Any] | None = None

    @model_validator(mode="after")
    def validate_action_fields(self):
        if self.action == "answer":
            if not self.final_answer:
                raise ValueError("final_answer is required when action='answer'")
            if self.tool_name is not None or self.tool_arguments is not None:
                raise ValueError("Tool fields must be empty when action='answer'")

        if self.action == "use_tool":
            if self.tool_name is None or self.tool_arguments is None:
                raise ValueError(
                    "tool_name and tool_arguments are required when action='use_tool'"
                )
            if self.final_answer is not None:
                raise ValueError("final_answer must be empty when action='use_tool'")

        return self

In [ ]:
answer_decision = AgentDecision.model_validate(
    {
        "action": "answer",
        "reason": "The requested concept is available in the current context.",
        "confidence": 0.91,
        "final_answer": "A tool registry is an explicit allowlist of executable capabilities.",
    }
)

tool_decision = AgentDecision.model_validate(
    {
        "action": "use_tool",
        "reason": "The calculation should be executed by the registered arithmetic tool.",
        "confidence": 0.98,
        "tool_name": "multiply",
        "tool_arguments": {
            "first_number": 24,
            "second_number": 17,
        },
    }
)

print(answer_decision.model_dump())
print(tool_decision.model_dump())

{'action': 'answer', 'reason': 'The requested concept is available in the current context.', 'confidence': 0.91, 'final_answer': 'A tool registry is an explicit allowlist of executable capabilities.', 'tool_name': None, 'tool_arguments': None}
{'action': 'use_tool', 'reason': 'The calculation should be executed by the registered arithmetic tool.', 'confidence': 0.98, 'final_answer': None, 'tool_name': 'multiply', 'tool_arguments': {'first_number': 24, 'second_number': 17}}


This model prevents contradictory states such as selecting `answer` while also requesting a tool. Later, we will improve the design further by using separate models for final answers and tool requests, but the conditional validator makes the underlying rule easy to see.

## 6. Parse JSON returned by a model

Model APIs frequently return JSON text or provide a structured-output feature that produces data compatible with a supplied schema. When the result arrives as text, `model_validate_json` combines JSON parsing with schema validation.

In [ ]:
raw_model_output = (
    '{'
    '"action": "use_tool",'
    '"reason": "The arithmetic operation should be handled by the calculator tool.",'
    '"confidence": 0.96,'
    '"tool_name": "multiply",'
    '"tool_arguments": {'
    '"first_number": 12,'
    '"second_number": 9'
    '}'
    '}'
)

parsed_decision = AgentDecision.model_validate_json(raw_model_output)

print(type(parsed_decision).__name__)
print(parsed_decision.model_dump())

AgentDecision
{'action': 'use_tool', 'reason': 'The arithmetic operation should be handled by the calculator tool.', 'confidence': 0.96, 'final_answer': None, 'tool_name': 'multiply', 'tool_arguments': {'first_number': 12, 'second_number': 9}}


JSON parsing and application validation remain distinct even though one method performs both operations. Malformed JSON fails during parsing, while well-formed JSON with unsupported values fails during schema validation.

In [ ]:
invalid_json_outputs = [
    # Invalid JSON syntax: the closing brace is missing.
    '{"action": "answer"',

    # Valid JSON syntax, invalid application data.
    (
        '{'
        '"action": "answer",'
        '"reason": "The context contains enough information to answer directly.",'
        '"confidence": 0.90'
        '}'
    ),
]

for raw_output in invalid_json_outputs:
    try:
        AgentDecision.model_validate_json(raw_output)
    except ValidationError as error:
        first_error = error.errors()[0]
        print(first_error["type"], "→", first_error["msg"])

json_invalid → Invalid JSON: EOF while parsing an object at line 1 column 19
value_error → Value error, final_answer is required when action='answer'


## 7. Generate JSON Schema

Pydantic can generate a standard JSON Schema from a model. Model providers that support structured outputs or native tool calling can use a related schema to constrain the generated response.

A schema improves structural reliability, but it does not prove that the model selected the right action or supplied factually correct arguments. Semantic correctness still requires application rules, evaluation, and sometimes human approval.

In [ ]:
import json


decision_schema = AgentDecision.model_json_schema()
print(json.dumps(decision_schema, indent=2))

{
  "additionalProperties": false,
  "properties": {
    "action": {
      "enum": [
        "answer",
        "use_tool"
      ],
      "title": "Action",
      "type": "string"
    },
    "reason": {
      "minLength": 10,
      "title": "Reason",
      "type": "string"
    },
    "confidence": {
      "maximum": 1.0,
      "minimum": 0.0,
      "title": "Confidence",
      "type": "number"
    },
    "final_answer": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Final Answer"
    },
    "tool_name": {
      "anyOf": [
        {
          "const": "multiply",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Tool Name"
    },
    "tool_arguments": {
      "anyOf": [
        {
          "additionalProperties": true,
          "type": "object"
        },
        {
          "type": "null"
      

The generated schema contains property definitions, allowed literal values, required fields, and restrictions on additional properties. In the next notebook, we will use schemas like this to describe tools in a provider-compatible format.

## 8. Validation is necessary, but not sufficient

A schema can guarantee that `confidence` is a number between zero and one. It cannot guarantee that the confidence is calibrated. It can guarantee that the selected tool is called `multiply`. It cannot guarantee that multiplication is the correct operation for the user's task.

It is helpful to separate four layers of checking:

| Layer | Example question |
|---|---|
| Syntax | Is this valid JSON? |
| Schema | Are the required fields and types correct? |
| Policy | Is this tool permitted for this user and situation? |
| Semantics | Is this action actually appropriate for the task? |

Pydantic primarily addresses the schema layer. A reliable agent needs all four.

## 9. Exercise: validate a division tool request

Create two models:

1. `DivideArguments` containing `numerator` and `denominator` as floats;
2. `DivideRequest` containing the literal tool name `"divide"` and nested `DivideArguments`.

Add a validator that rejects zero as the denominator. Configure both models to reject extra fields. Finally, validate the example request and print the result of the division.

In [ ]:
class DivideArguments(BaseModel):
    # Configure the model and define the two numeric fields.
    # Add a validator that rejects denominator == 0.
    pass


class DivideRequest(BaseModel):
    # Define tool_name as Literal["divide"] and add nested arguments.
    pass


exercise_request = {
    "tool_name": "divide",
    "arguments": {
        "numerator": 81,
        "denominator": 9,
    },
}

# Validate exercise_request and print the calculated result.

<details>
<summary><strong>Reveal one possible solution</strong></summary>

```python
from pydantic import field_validator


class DivideArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    numerator: float
    denominator: float

    @field_validator("denominator")
    @classmethod
    def denominator_must_not_be_zero(cls, value: float) -> float:
        if value == 0:
            raise ValueError("denominator must not be zero")
        return value


class DivideRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    tool_name: Literal["divide"]
    arguments: DivideArguments


validated_request = DivideRequest.model_validate(exercise_request)
result = (
    validated_request.arguments.numerator
    / validated_request.arguments.denominator
)
print(result)
```

The result should be `9.0`. Also test the validator with a denominator of zero and confirm that validation fails before division is attempted.

</details>

## Notebook summary

We replaced loosely structured dictionaries with explicit Pydantic contracts. We constrained action names and numeric ranges, rejected unexpected fields, validated nested tool arguments, expressed conditional rules, parsed JSON model output, and generated JSON Schema.

In the next notebook, we will design a small but reusable tool system. Each tool will combine executable code with a description and an argument schema, allowing the application to expose capabilities to a real model API without giving the model unrestricted access to Python.